In [ ]:
# FireWatch AI: Dataset Preparation

In this notebook, I am preparing the fire/smoke and PPE datasets for the training stages of my FireWatch AI project.

I have already inspected the datasets and identified that some annotation files contain polygon labels instead of standard YOLO bounding-box labels. Since my project uses YOLO for object detection, I need to make sure the candidate datasets contain valid detection labels before training.

I will work with the existing candidate folders in Google Drive. I will preserve the original datasets, avoid copying all images again, and focus only on the checks needed to confirm dataset readiness.

By the end of this notebook, I want to know whether both candidate datasets are ready for training or whether any specific preparation is still required.

In [ ]:
from pathlib import Path
from google.colab import drive

# I am connecting Google Drive so I can access my existing project files.
drive.mount("/content/gdrive_reconnect", force_remount=False)

PROJECT_ROOT = Path("/content/gdrive_reconnect/MyDrive/FireWatch-AI")
DATASET_ROOT = PROJECT_ROOT / "datasets"

CANDIDATES = {
    "Fire and Smoke": DATASET_ROOT / "candidates" / "fire_smoke_v2_bbox_candidate",
    "PPE Compliance": DATASET_ROOT / "candidates" / "ppe_compliance_v2_bbox_candidate",
}

print("Project folder exists:", PROJECT_ROOT.is_dir())
print("Dataset folder exists:", DATASET_ROOT.is_dir())

if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project folder not found: {PROJECT_ROOT}\n"
        "Check the mounted Google Drive path before continuing."
    )

Mounted at /content/gdrive_reconnect
Project folder exists: True
Dataset folder exists: True


In [ ]:
## Step 1: Check the Existing Candidate Datasets

Before preparing labels, I will inspect the existing candidate folders and their `data.yaml` files.

This is important because the original datasets must remain unchanged, and I should not start training if the candidate folders are incomplete or their image paths are incorrect.

I will check the configuration and expected train, validation, and test image directories. I will not repeat the earlier full dataset audit.

In [ ]:
from pathlib import Path

DATASET_ROOT = PROJECT_ROOT / "datasets"

# I am locating the existing YAML files so I can use
# the actual dataset configuration instead of guessing paths.
for folder_name in [
    "fire_smoke_v2",
    "ppe_compliance_v2",
    "candidates/fire_smoke_v2_bbox_candidate",
    "candidates/ppe_compliance_v2_bbox_candidate",
]:
    folder = DATASET_ROOT / folder_name

    print(f"\n{'=' * 55}")
    print("Folder:", folder)
    print("Exists:", folder.is_dir())

    if folder.is_dir():
        yaml_files = list(folder.rglob("*.yaml"))
        yaml_files += list(folder.rglob("*.yml"))

        if yaml_files:
            print("YAML files found:")
            for yaml_file in yaml_files:
                print(" ", yaml_file)
        else:
            print("No YAML files found inside this folder.")


Folder: /content/gdrive_reconnect/MyDrive/FireWatch-AI/datasets/fire_smoke_v2
Exists: True
YAML files found:
  /content/gdrive_reconnect/MyDrive/FireWatch-AI/datasets/fire_smoke_v2/data.yaml

Folder: /content/gdrive_reconnect/MyDrive/FireWatch-AI/datasets/ppe_compliance_v2
Exists: True
YAML files found:
  /content/gdrive_reconnect/MyDrive/FireWatch-AI/datasets/ppe_compliance_v2/data.yaml

Folder: /content/gdrive_reconnect/MyDrive/FireWatch-AI/datasets/candidates/fire_smoke_v2_bbox_candidate
Exists: True
No YAML files found inside this folder.

Folder: /content/gdrive_reconnect/MyDrive/FireWatch-AI/datasets/candidates/ppe_compliance_v2_bbox_candidate
Exists: True
No YAML files found inside this folder.


In [ ]:
from pathlib import Path
import yaml

DATASET_ROOT = PROJECT_ROOT / "datasets"

DATASET_PAIRS = {
    "Fire and Smoke": {
        "original": DATASET_ROOT / "fire_smoke_v2",
        "candidate": DATASET_ROOT / "candidates" / "fire_smoke_v2_bbox_candidate",
    },
    "PPE Compliance": {
        "original": DATASET_ROOT / "ppe_compliance_v2",
        "candidate": DATASET_ROOT / "candidates" / "ppe_compliance_v2_bbox_candidate",
    },
}

for name, paths in DATASET_PAIRS.items():
    print(f"\n{'=' * 55}")
    print(name)

    original_yaml = paths["original"] / "data.yaml"

    # I read the original configuration without changing it.
    with original_yaml.open("r", encoding="utf-8") as file:
        config = yaml.safe_load(file)

    print("Original classes:", config.get("names"))
    print("Original class count:", config.get("nc"))

    candidate = paths["candidate"]

    # I check the expected split folders in the candidate.
    for split in ("train", "valid", "val", "test"):
        split_dir = candidate / split
        if split_dir.exists():
            print(f"{split}/ exists")

            for subfolder in ("images", "labels"):
                path = split_dir / subfolder
                if path.exists():
                    print(f"  {subfolder}/ exists")
                    print(
                        f"  {subfolder} file count:",
                        sum(1 for item in path.iterdir() if item.is_file())
                    )

    # I also check whether the candidate contains nested split folders.
    print("Top-level entries:", [p.name for p in candidate.iterdir()])


Fire and Smoke
Original classes: ['fire', 'smoke']
Original class count: 2
train/ exists
  images/ exists
  images file count: 3320
  labels/ exists
  labels file count: 3320
valid/ exists
  images/ exists
  images file count: 684
  labels/ exists
  labels file count: 379
test/ exists
  images/ exists
  images file count: 300
  labels/ exists
  labels file count: 300
Top-level entries: ['README.roboflow.txt', 'README.dataset.txt', 'valid', 'test', 'train']

PPE Compliance
Original classes: ['Boots', 'Gloves', 'Goggles', 'Helmet', 'Mask', 'NO-Boots', 'NO-Gloves', 'NO-Goggles', 'NO-Helmet', 'NO-Mask', 'NO-Vest', 'Person', 'Vest']
Original class count: 13
train/ exists
  images/ exists
  images file count: 1752
test/ exists
  images/ exists
  images file count: 482
  labels/ exists
  labels file count: 482
Top-level entries: ['test', 'README.roboflow.txt', 'README.dataset.txt', 'train']


In [ ]:
from pathlib import Path

DATASET_ROOT = PROJECT_ROOT / "datasets"

ORIGINAL_DATASETS = {
    "Fire and Smoke": DATASET_ROOT / "fire_smoke_v2",
    "PPE Compliance": DATASET_ROOT / "ppe_compliance_v2",
}

for name, dataset_path in ORIGINAL_DATASETS.items():
    print(f"\n{'=' * 55}")
    print(name)

    for split in ("train", "valid", "val", "test"):
        split_path = dataset_path / split

        if not split_path.is_dir():
            continue

        images_path = split_path / "images"
        labels_path = split_path / "labels"

        image_count = (
            sum(1 for p in images_path.iterdir() if p.is_file())
            if images_path.is_dir() else 0
        )
        label_count = (
            sum(1 for p in labels_path.glob("*.txt") if p.is_file())
            if labels_path.is_dir() else 0
        )

        print(
            f"{split}: "
            f"images={image_count}, "
            f"label_files={label_count}, "
            f"labels_folder_exists={labels_path.is_dir()}"
        )


Fire and Smoke
train: images=3320, label_files=3320, labels_folder_exists=True
valid: images=684, label_files=684, labels_folder_exists=True
test: images=300, label_files=300, labels_folder_exists=True

PPE Compliance
train: images=7721, label_files=7721, labels_folder_exists=True
valid: images=1460, label_files=1460, labels_folder_exists=True
test: images=482, label_files=482, labels_folder_exists=True


In [ ]:
## Step 2: Prepare the Existing Candidate Datasets

I have confirmed that the original fire/smoke and PPE datasets contain their expected training, validation, and test images and label files.

However, the existing candidate folders are incomplete. The fire/smoke candidate has a validation-label count mismatch, and the PPE candidate is missing its validation split and some training labels. Both candidates are also missing their own `data.yaml` files.

Instead of copying all images again, I will prepare the candidate datasets by reusing the existing images and creating a separate candidate label structure. I will preserve the original datasets and their annotations.

Before making any changes, I will check whether the candidate image files correspond to the original image files. This will help me avoid overwriting or pairing labels with the wrong images.

In [ ]:
### Step 3: Compare Original and Candidate Images

In this step, I will compare the image filenames in each original split with the corresponding candidate split.

This check helps identify whether the candidate folders contain all expected images. It does not change any files and does not assume that the candidate labels are already correct.

In [ ]:
from pathlib import Path

# I compare image filenames without copying or modifying any files.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

SPLIT_NAMES = ("train", "valid", "test")

for dataset_name, paths in DATASET_PAIRS.items():
    original_root = paths["original"]
    candidate_root = paths["candidate"]

    print(f"\n{'=' * 60}")
    print(dataset_name)

    for split in SPLIT_NAMES:
        original_images_dir = original_root / split / "images"
        candidate_images_dir = candidate_root / split / "images"

        if not original_images_dir.is_dir():
            print(f"{split}: original image folder missing")
            continue

        original_names = {
            p.name for p in original_images_dir.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        }

        if not candidate_images_dir.is_dir():
            print(
                f"{split}: candidate image folder missing; "
                f"{len(original_names)} original images available"
            )
            continue

        candidate_names = {
            p.name for p in candidate_images_dir.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        }

        missing_names = original_names - candidate_names
        extra_names = candidate_names - original_names

        print(
            f"{split}: original={len(original_names)}, "
            f"candidate={len(candidate_names)}, "
            f"missing={len(missing_names)}, extra={len(extra_names)}"
        )


Fire and Smoke
train: original=3320, candidate=3320, missing=0, extra=0
valid: original=684, candidate=684, missing=0, extra=0
test: original=300, candidate=300, missing=0, extra=0

PPE Compliance
train: original=7721, candidate=1752, missing=5969, extra=0
valid: candidate image folder missing; 1460 original images available
test: original=482, candidate=482, missing=0, extra=0


In [ ]:
### Step 4: Choose a Safe Preparation Method

After checking image coverage, I will determine the safest way to prepare the candidate datasets.

The goal is to reuse the existing images, preserve the original labels, and convert polygon annotations into standard YOLO detection bounding boxes only where needed.

I will not automatically copy the entire dataset, overwrite original labels, or create a training configuration until the candidate image and label structure has been confirmed.

In [ ]:
### Step 5: Repair the Candidate Dataset Structure

I have confirmed that the fire/smoke candidate contains all original images, while the PPE candidate is missing part of its training split and its validation split.

I will now prepare a repair plan that reuses the existing candidate images and restores missing images from the original dataset only where necessary. I will keep original annotations unchanged and create or update candidate labels separately.

Before training, I will also ensure that each candidate has the correct class configuration and standard YOLO detection labels. I will not consider a candidate ready merely because its image counts match.

In [ ]:
### Step 6: Restore Missing Candidate Files

I will restore missing images and labels from the original datasets into the candidate folders. Existing candidate images will be kept, and original dataset files will only be read.

This step will not overwrite existing candidate images or modify original annotations. After restoration, I will convert polygon rows in the candidate labels into bounding boxes in a separate preparation step and verify the results before training.

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/gdrive_reconnect/MyDrive/FireWatch-AI/datasets")

for name, src_name, dst_name in [
    ("Fire/Smoke", "fire_smoke_v2", "fire_smoke_v2_bbox_candidate"),
    ("PPE", "ppe_compliance_v2", "ppe_compliance_v2_bbox_candidate"),
]:
    src = root / src_name
    dst = root / "candidates" / dst_name

    for split in ["train", "valid", "test"]:
        for kind, exts in [
            ("images", {".jpg", ".jpeg", ".png", ".bmp", ".webp"}),
            ("labels", {".txt"}),
        ]:
            a, b = src / split / kind, dst / split / kind
            if not a.is_dir():
                continue
            b.mkdir(parents=True, exist_ok=True)
            for f in a.iterdir():
                target = b / f.name
                if f.is_file() and f.suffix.lower() in exts and not target.exists():
                    shutil.copy2(f, target)

    print(f"{name}: missing files restored")

Fire/Smoke: missing files restored
PPE: missing files restored


In [ ]:
### Step 7: Convert Candidate Labels and Create Dataset Configurations

I have restored the missing candidate files. Now I will prepare the candidate annotations for YOLO object detection.

Existing five-value bounding-box labels will remain unchanged. Polygon labels will be converted to bounding boxes in the candidate folders only, with backups saved separately. I will then create a `data.yaml` file for each dataset using its original class names.

The original datasets will remain unchanged. This step prepares the data, but training readiness must still be verified.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=True)

# Locate the existing project instead of assuming its path.
for base in [
    Path("/content/drive/MyDrive/FireWatch-AI"),
    Path("/content/drive/MyDrive"),
]:
    if base.name == "MyDrive":
        matches = list(base.glob("**/FireWatch-AI"))
        matches = [p for p in matches if p.is_dir()]
    else:
        matches = [base] if base.is_dir() else []

    for project in matches:
        print("PROJECT FOUND:", project)
        for name in ["fire_smoke_v2", "ppe_compliance_v2"]:
            folder = project / "datasets" / name
            print(name, "exists:", folder.is_dir(),
                  "| YAML:", (folder / "data.yaml").is_file())

Mounted at /content/drive
PROJECT FOUND: /content/drive/MyDrive/FireWatch-AI
fire_smoke_v2 exists: True | YAML: True
ppe_compliance_v2 exists: True | YAML: True
PROJECT FOUND: /content/drive/MyDrive/FireWatch-AI
fire_smoke_v2 exists: True | YAML: True
ppe_compliance_v2 exists: True | YAML: True


In [ ]:
### Step 8: Convert Polygon Annotations into Bounding Boxes

I have reconnected Google Drive and confirmed the original dataset paths. I will now prepare the candidate annotations for YOLO object detection.

I will keep existing bounding-box labels unchanged, convert polygon annotations into their enclosing bounding boxes, and save backups of affected candidate label files. I will also create a separate `data.yaml` for each candidate dataset.

The original datasets will remain unchanged. After this step, I will verify the candidate configuration and label formats before training.

In [ ]:
from pathlib import Path
import shutil
import yaml

root = Path("/content/drive/MyDrive/FireWatch-AI/datasets")

for original_name, candidate_name in [
    ("fire_smoke_v2", "fire_smoke_v2_bbox_candidate"),
    ("ppe_compliance_v2", "ppe_compliance_v2_bbox_candidate"),
]:
    original = root / original_name
    candidate = root / "candidates" / candidate_name

    with (original / "data.yaml").open(encoding="utf-8") as f:
        config = yaml.safe_load(f)

    backup_root = candidate / "_polygon_label_backups"
    converted = 0

    for split in ("train", "valid", "test"):
        labels_dir = candidate / split / "labels"
        if not labels_dir.is_dir():
            continue

        for label_file in labels_dir.glob("*.txt"):
            lines = label_file.read_text(encoding="utf-8").splitlines()
            output, changed = [], False

            for line in lines:
                values = line.split()
                if not values:
                    continue

                if len(values) == 5:
                    output.append(line)
                    continue

                if len(values) < 7 or len(values[1:]) % 2 != 0:
                    raise ValueError(f"Unexpected label format: {label_file}")

                class_id = int(values[0])
                coords = [float(v) for v in values[1:]]

                if not 0 <= class_id < config["nc"]:
                    raise ValueError(f"Invalid class ID in {label_file}")
                if not all(0 <= v <= 1 for v in coords):
                    raise ValueError(f"Invalid normalized coordinates: {label_file}")

                xs, ys = coords[0::2], coords[1::2]
                xmin, xmax = min(xs), max(xs)
                ymin, ymax = min(ys), max(ys)

                output.append(
                    f"{class_id} {(xmin+xmax)/2:.6f} "
                    f"{(ymin+ymax)/2:.6f} "
                    f"{xmax-xmin:.6f} {ymax-ymin:.6f}"
                )
                changed = True

            if changed:
                backup = backup_root / split / label_file.name
                backup.parent.mkdir(parents=True, exist_ok=True)

                if not backup.exists():
                    shutil.copy2(label_file, backup)

                label_file.write_text("\n".join(output) + "\n", encoding="utf-8")
                converted += 1

    candidate_config = {
        "path": str(candidate),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "names": config["names"],
        "nc": config["nc"],
    }

    with (candidate / "data.yaml").open("w", encoding="utf-8") as f:
        yaml.safe_dump(candidate_config, f, sort_keys=False)

    print(f"{candidate_name}: converted label files = {converted}")
    print(f"Configuration saved: {candidate / 'data.yaml'}")

fire_smoke_v2_bbox_candidate: converted label files = 5
Configuration saved: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/fire_smoke_v2_bbox_candidate/data.yaml
ppe_compliance_v2_bbox_candidate: converted label files = 4
Configuration saved: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/ppe_compliance_v2_bbox_candidate/data.yaml


In [ ]:
### Step 9 : Verify the Prepared Datasets

I have converted the polygon annotations found in the candidate label files and created candidate dataset configurations.

Now I will verify that each configured image directory exists, image and label counts match, and no polygon-like annotation rows remain. This is the final preparation check before I move to model training.

I will not modify any files during this check.

In [ ]:
from pathlib import Path
import yaml

root = Path("/content/drive/MyDrive/FireWatch-AI/datasets/candidates")

for name in ["fire_smoke_v2_bbox_candidate", "ppe_compliance_v2_bbox_candidate"]:
    folder = root / name
    config_path = folder / "data.yaml"

    print(f"\n{'=' * 45}\n{name}")
    with config_path.open(encoding="utf-8") as f:
        config = yaml.safe_load(f)

    ready = True

    for split in ["train", "val", "test"]:
        image_dir = folder / config[split] / "images" if False else None
        split_name = "valid" if split == "val" else split
        image_dir = folder / split_name / "images"
        label_dir = folder / split_name / "labels"

        if not image_dir.is_dir() or not label_dir.is_dir():
            print(f"{split}: MISSING IMAGE OR LABEL FOLDER")
            ready = False
            continue

        images = [p for p in image_dir.iterdir() if p.is_file()]
        labels = list(label_dir.glob("*.txt"))

        polygons = 0
        for label in labels:
            for line in label.read_text(encoding="utf-8").splitlines():
                values = line.split()
                if values and len(values) != 5:
                    polygons += 1

        print(
            f"{split}: images={len(images)}, labels={len(labels)}, "
            f"non-5-value rows={polygons}"
        )

        if len(images) != len(labels) or polygons:
            ready = False

    print("PRELIMINARY STATUS:", "PASS" if ready else "NOT READY")


fire_smoke_v2_bbox_candidate
train: images=3320, labels=3320, non-5-value rows=0
val: images=684, labels=684, non-5-value rows=0
test: images=300, labels=300, non-5-value rows=0
PRELIMINARY STATUS: PASS

ppe_compliance_v2_bbox_candidate
train: MISSING IMAGE OR LABEL FOLDER
val: MISSING IMAGE OR LABEL FOLDER
test: images=482, labels=482, non-5-value rows=0
PRELIMINARY STATUS: NOT READY


In [ ]:
### Step 10: Complete the PPE Candidate Dataset

My fire/smoke candidate has passed the preliminary readiness check. The PPE candidate still needs its training and validation folders restored.

I will reuse the original PPE dataset to restore missing files without overwriting existing candidate files or modifying the original dataset. After this, I will verify the PPE split structure again.

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/FireWatch-AI/datasets")
src = root / "ppe_compliance_v2"
dst = root / "candidates/ppe_compliance_v2_bbox_candidate"

for split in ["train", "valid", "test"]:
    print(f"\n{split.upper()}")

    for kind in ["images", "labels"]:
        source = src / split / kind
        target = dst / split / kind

        source_count = sum(1 for p in source.iterdir() if p.is_file()) if source.is_dir() else 0
        target_count = sum(1 for p in target.iterdir() if p.is_file()) if target.is_dir() else 0

        print(f"{kind}: original={source_count}, candidate={target_count}, missing={max(0, source_count-target_count)}")


TRAIN
images: original=7721, candidate=7721, missing=0
labels: original=7721, candidate=79, missing=7642

VALID
images: original=1460, candidate=0, missing=1460
labels: original=1460, candidate=0, missing=1460

TEST
images: original=482, candidate=482, missing=0
labels: original=482, candidate=482, missing=0


In [ ]:
### Step 10.1: Restore Missing PPE Labels Efficiently

I have confirmed that the PPE candidate already contains all training and test images, but many training labels and all validation files are still missing.

I will restore only missing label files from the original PPE dataset. Existing candidate labels will remain unchanged, and the original dataset will not be modified.

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/drive/MyDrive/FireWatch-AI/datasets")
src = root / "ppe_compliance_v2"
dst = root / "candidates/ppe_compliance_v2_bbox_candidate"

for split in ["train", "valid"]:
    source = src / split / "labels"
    target = dst / split / "labels"
    target.mkdir(parents=True, exist_ok=True)

    missing = [p for p in source.glob("*.txt") if not (target / p.name).exists()]

    for file in missing:
        shutil.copy2(file, target / file.name)

    print(f"{split}: restored {len(missing)} label files")

train: restored 7642 label files
valid: restored 1460 label files


In [ ]:
### Step 10.2: Convert PPE Polygon Labels and Verify Dataset Readiness

I have restored the missing PPE training and validation labels. Now I will convert any remaining polygon annotations in the candidate dataset into bounding boxes, preserve backups of the affected candidate labels, and verify the dataset configuration before training.

The original PPE dataset will remain unchanged.

In [ ]:
from pathlib import Path
from collections import Counter

labels_root = Path(
    "/content/drive/MyDrive/FireWatch-AI/datasets/"
    "candidates/ppe_compliance_v2_bbox_candidate"
)

for split in ["train", "valid", "test"]:
    counts = Counter()
    folder = labels_root / split / "labels"

    for file in folder.glob("*.txt") if folder.is_dir() else []:
        for line in file.read_text(encoding="utf-8").splitlines():
            values = line.split()
            if values:
                counts["bbox_rows" if len(values) == 5 else "other_rows"] += 1

    print(f"{split}: {dict(counts)}")

train: {'bbox_rows': 31599, 'other_rows': 78}
valid: {'bbox_rows': 5830, 'other_rows': 24}
test: {'bbox_rows': 1978}


In [ ]:
### Step 10.3: Convert Remaining PPE Polygon Annotations

I have identified 78 polygon rows in the training labels and 24 in the validation labels. I will convert these rows into their enclosing YOLO bounding boxes in the candidate dataset only. Existing bounding-box rows will remain unchanged, and I will preserve backups of affected candidate label files.

After conversion, I will verify the label format and dataset configuration.

In [ ]:
from pathlib import Path
import shutil
import yaml

root = Path("/content/drive/MyDrive/FireWatch-AI/datasets")
original = root / "ppe_compliance_v2"
candidate = root / "candidates/ppe_compliance_v2_bbox_candidate"

with (original / "data.yaml").open(encoding="utf-8") as f:
    config = yaml.safe_load(f)

backup_root = candidate / "_polygon_label_backups"
converted_rows = 0

for split in ["train", "valid", "test"]:
    folder = candidate / split / "labels"
    if not folder.is_dir():
        continue

    for file in folder.glob("*.txt"):
        lines = file.read_text(encoding="utf-8").splitlines()
        output, changed = [], False

        for line in lines:
            values = line.split()
            if not values:
                continue
            if len(values) == 5:
                output.append(line)
                continue
            if len(values) < 7 or len(values[1:]) % 2:
                raise ValueError(f"Unexpected label format: {file}")

            cls = int(values[0])
            coords = list(map(float, values[1:]))
            if not 0 <= cls < config["nc"] or not all(0 <= v <= 1 for v in coords):
                raise ValueError(f"Invalid annotation: {file}")

            xs, ys = coords[0::2], coords[1::2]
            x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
            output.append(
                f"{cls} {(x1+x2)/2:.6f} {(y1+y2)/2:.6f} "
                f"{x2-x1:.6f} {y2-y1:.6f}"
            )
            changed = True
            converted_rows += 1

        if changed:
            backup = backup_root / split / file.name
            backup.parent.mkdir(parents=True, exist_ok=True)
            if not backup.exists():
                shutil.copy2(file, backup)
            file.write_text("\n".join(output) + "\n", encoding="utf-8")

config.update({
    "path": str(candidate),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
})
with (candidate / "data.yaml").open("w", encoding="utf-8") as f:
    yaml.safe_dump(config, f, sort_keys=False)

print("Polygon rows converted:", converted_rows)
print("PPE candidate YAML updated.")

Polygon rows converted: 102
PPE candidate YAML updated.


In [ ]:
### Step 11: Final Dataset Readiness Check

I have prepared the fire/smoke and PPE candidate datasets and converted the polygon annotations into bounding boxes.

In this final check, I will verify that the configured image directories exist, every image has a corresponding label file, all annotation rows follow the five-value YOLO detection format, and class IDs and normalized coordinates are valid.

This check is read-only. I will use the results to decide whether the datasets are ready for the training stage.

In [ ]:
from pathlib import Path
import yaml

project = Path("/content/drive/MyDrive/FireWatch-AI")
candidates = project / "datasets" / "candidates"

datasets = [
    candidates / "fire_smoke_v2_bbox_candidate",
    candidates / "ppe_compliance_v2_bbox_candidate",
]

image_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for folder in datasets:
    print(f"\n{'=' * 55}\n{folder.name}")
    yaml_path = folder / "data.yaml"

    if not yaml_path.is_file():
        print("NOT READY: data.yaml missing")
        continue

    config = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))
    names = config["names"]
    nc = config["nc"]
    issues = []

    if len(names) != nc:
        issues.append("Class names count does not match nc")

    for split in ["train", "val", "test"]:
        image_rel = config.get(split)
        if not image_rel:
            issues.append(f"{split}: path missing from YAML")
            continue

        image_dir = Path(config.get("path", folder)) / image_rel
        label_dir = folder / ("valid" if split == "val" else split) / "labels"

        if not image_dir.is_dir() or not label_dir.is_dir():
            issues.append(f"{split}: image or label folder missing")
            print(f"{split}: MISSING FOLDER")
            continue

        images = {
            p.stem for p in image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in image_exts
        }
        labels = {p.stem for p in label_dir.glob("*.txt")}

        missing_labels = images - labels
        extra_labels = labels - images
        bad_rows = 0

        for label_file in label_dir.glob("*.txt"):
            for line in label_file.read_text(encoding="utf-8").splitlines():
                values = line.split()
                if not values:
                    continue
                try:
                    nums = list(map(float, values))
                    valid = (
                        len(nums) == 5
                        and nums[0].is_integer()
                        and 0 <= int(nums[0]) < nc
                        and all(0 <= v <= 1 for v in nums[1:])
                        and nums[3] > 0 and nums[4] > 0
                    )
                except ValueError:
                    valid = False

                if not valid:
                    bad_rows += 1

        print(
            f"{split}: images={len(images)}, labels={len(labels)}, "
            f"missing_labels={len(missing_labels)}, "
            f"extra_labels={len(extra_labels)}, invalid_rows={bad_rows}"
        )

        if missing_labels or extra_labels or bad_rows:
            issues.append(f"{split}: label or annotation issue")

    print("FINAL STATUS:", "PASS" if not issues else "NOT READY")
    for issue in issues:
        print(" -", issue)


fire_smoke_v2_bbox_candidate
train: images=3320, labels=3320, missing_labels=0, extra_labels=0, invalid_rows=0
val: images=684, labels=684, missing_labels=0, extra_labels=0, invalid_rows=0
test: images=300, labels=300, missing_labels=0, extra_labels=0, invalid_rows=0
FINAL STATUS: PASS

ppe_compliance_v2_bbox_candidate
train: images=7721, labels=7721, missing_labels=0, extra_labels=0, invalid_rows=0
val: MISSING FOLDER
test: images=482, labels=482, missing_labels=0, extra_labels=0, invalid_rows=0
FINAL STATUS: NOT READY
 - val: image or label folder missing


In [ ]:
### Step 12: Fix the PPE Validation Configuration

My fire/smoke candidate has passed the final readiness check. The PPE candidate has valid training and test splits, but its validation path is not resolving correctly.

I will inspect the existing PPE validation folders and update only the candidate configuration if needed. I will not copy the dataset again or modify the original dataset.

In [ ]:
from pathlib import Path
import yaml

root = Path("/content/drive/MyDrive/FireWatch-AI/datasets")
original = root / "ppe_compliance_v2"
candidate = root / "candidates/ppe_compliance_v2_bbox_candidate"

for folder in [original, candidate]:
    print(f"\n{folder.name}")
    for split in ["valid", "val"]:
        for kind in ["images", "labels"]:
            path = folder / split / kind
            print(f"{split}/{kind}: exists={path.is_dir()}")

            if path.is_dir():
                print("  files:", sum(1 for p in path.iterdir() if p.is_file()))

config = yaml.safe_load((candidate / "data.yaml").read_text(encoding="utf-8"))
print("\nCurrent YAML validation path:", config.get("val"))


ppe_compliance_v2
valid/images: exists=True
  files: 1460
valid/labels: exists=True
  files: 1460
val/images: exists=False
val/labels: exists=False

ppe_compliance_v2_bbox_candidate
valid/images: exists=True
  files: 1460
valid/labels: exists=True
  files: 1460
val/images: exists=False
val/labels: exists=False

Current YAML validation path: valid/images


In [ ]:
### Step 13: Restore Missing PPE Validation Images

I found that the original PPE validation split contains 1,460 images and labels. The candidate already has the validation labels, but its validation image folder is missing.

I will restore only the missing validation images from the original dataset, without changing the original annotations or copying other splits again.

In [ ]:
from pathlib import Path
import shutil

root = Path("/content/drive/MyDrive/FireWatch-AI/datasets")
src = root / "ppe_compliance_v2/valid/images"
dst = root / "candidates/ppe_compliance_v2_bbox_candidate/valid/images"
dst.mkdir(parents=True, exist_ok=True)

missing = [p for p in src.iterdir() if p.is_file() and not (dst / p.name).exists()]

for p in missing:
    shutil.copy2(p, dst / p.name)

print(f"Restored {len(missing)} validation images.")
print(f"Candidate validation images: {sum(1 for p in dst.iterdir() if p.is_file())}")

Restored 1460 validation images.
Candidate validation images: 1460


In [ ]:
### Step 13: Confirm Dataset Readiness

I have restored the missing PPE validation images. I will run the final readiness check again to confirm that both datasets have valid split paths, matching image and label counts, and valid YOLO detection labels.

Once both candidates pass, I can move to fire/smoke model training.

In [ ]:
### Step 14: Final Dataset Readiness Check

I have prepared the fire/smoke and PPE candidate datasets. I will now verify that all three splits contain matching images and labels, every annotation uses the standard five-value YOLO detection format, class IDs and normalized coordinates are valid, and the dataset configuration points to existing image folders.

This check only reads files. It does not modify the datasets.

In [ ]:
from pathlib import Path
import yaml

root = Path("/content/drive/MyDrive/FireWatch-AI/datasets/candidates")

datasets = [
    root / "fire_smoke_v2_bbox_candidate",
    root / "ppe_compliance_v2_bbox_candidate",
]

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

for folder in datasets:
    print(f"\n{'=' * 50}\n{folder.name}")
    yaml_path = folder / "data.yaml"
    issues = []

    if not yaml_path.is_file():
        print("FINAL STATUS: NOT READY | data.yaml missing")
        continue

    config = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))
    names, nc = config["names"], config["nc"]

    if len(names) != nc:
        issues.append("Class names count does not match nc")

    for split in ["train", "val", "test"]:
        image_dir = Path(config.get("path", folder)) / config.get(split, "")
        split_name = "valid" if split == "val" else split
        label_dir = folder / split_name / "labels"

        if not image_dir.is_dir() or not label_dir.is_dir():
            print(f"{split}: MISSING FOLDER")
            issues.append(f"{split}: missing image or label folder")
            continue

        images = {
            p.stem for p in image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in image_extensions
        }
        labels = {p.stem for p in label_dir.glob("*.txt")}

        missing = images - labels
        extra = labels - images
        invalid_rows = 0

        for label_file in label_dir.glob("*.txt"):
            for line in label_file.read_text(encoding="utf-8").splitlines():
                values = line.split()
                if not values:
                    continue

                try:
                    nums = list(map(float, values))
                    valid = (
                        len(nums) == 5
                        and nums[0].is_integer()
                        and 0 <= int(nums[0]) < nc
                        and all(0 <= x <= 1 for x in nums[1:])
                        and nums[3] > 0 and nums[4] > 0
                    )
                except ValueError:
                    valid = False

                if not valid:
                    invalid_rows += 1

        print(
            f"{split}: images={len(images)}, labels={len(labels)}, "
            f"missing={len(missing)}, extra={len(extra)}, "
            f"invalid_rows={invalid_rows}"
        )

        if missing or extra or invalid_rows:
            issues.append(f"{split}: label mismatch or invalid rows")

    print("FINAL STATUS:", "PASS" if not issues else "NOT READY")
    for issue in issues:
        print(" -", issue)


fire_smoke_v2_bbox_candidate
train: images=3320, labels=3320, missing=0, extra=0, invalid_rows=0
val: images=684, labels=684, missing=0, extra=0, invalid_rows=0
test: images=300, labels=300, missing=0, extra=0, invalid_rows=0
FINAL STATUS: PASS

ppe_compliance_v2_bbox_candidate
train: images=7721, labels=7721, missing=0, extra=0, invalid_rows=0
val: images=1460, labels=1460, missing=0, extra=0, invalid_rows=0
test: images=482, labels=482, missing=0, extra=0, invalid_rows=0
FINAL STATUS: PASS


In [ ]:
from google.colab import drive

# Connect Google Drive to this Colab session.
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/FireWatch-AI")

print("Drive connected:", Path("/content/drive/MyDrive").is_dir())
print("FireWatch AI folder exists:", PROJECT_ROOT.is_dir())

In [1]:
# I am packaging the prepared PPE dataset for Kaggle training.
# The original dataset will remain unchanged.

from pathlib import Path
import shutil

PROJECT_ROOT = Path("/content/drive/MyDrive/FireWatch-AI")

PPE_DATASET = (
    PROJECT_ROOT
    / "datasets/candidates/ppe_compliance_v2_bbox_candidate"

)

ZIP_BASE = PROJECT_ROOT / "datasets" / "ppe_compliance_v2_bbox_candidate"

if not PPE_DATASET.is_dir():
    raise FileNotFoundError(f"PPE dataset not found: {PPE_DATASET}")

# Create a ZIP archive of the prepared candidate dataset.
zip_path = shutil.make_archive(
    str(ZIP_BASE),
    "zip",
    root_dir=PPE_DATASET.parent,
    base_dir=PPE_DATASET.name
)

print("ZIP created:", zip_path)
print("ZIP size (GB):", round(Path(zip_path).stat().st_size / (1024**3), 2))
print("Original dataset preserved: YES")

FileNotFoundError: PPE dataset not found: /content/drive/MyDrive/FireWatch-AI/datasets/candidates/ppe_compliance_v2_bbox_candidate